In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import os

PROCESSED = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\data\processed"
RESULTS   = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\results"
MODELS    = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning\models"

# ── Load ──────────────────────────────────────────────────────────────────
X_test = np.load(os.path.join(PROCESSED, "X_test.npy"))

# Get feature names from cleaned EDA CSV
df_cols = pd.read_csv(os.path.join(PROCESSED, "cleaned_eda.csv"), nrows=1)
feature_names = [c for c in df_cols.columns if c != 'label']
print(f"Features: {len(feature_names)}")
print(f"X_test shape: {X_test.shape}")

X_test_df = pd.DataFrame(X_test, columns=feature_names)

# Load XGBoost
import xgboost as xgb
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(os.path.join(MODELS, "xgboost.pkl"))
print("XGBoost loaded.")

# ── SHAP ──────────────────────────────────────────────────────────────────
np.random.seed(42)
sample_idx = np.random.choice(len(X_test_df), 500, replace=False)
X_sample = X_test_df.iloc[sample_idx]

print("Computing SHAP values...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)
print(f"SHAP values shape: {np.array(shap_values).shape}")

# ── SHAP Bar Chart ────────────────────────────────────────────────────────
plt.style.use('dark_background')
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  plot_type="bar", max_display=10, show=False)
ax = plt.gca()
ax.set_facecolor('#1a0a2e')
plt.gcf().set_facecolor('#1a0a2e')
for label in ax.get_yticklabels():
    label.set_color('#CC99FF')
    label.set_fontsize(12)
for label in ax.get_xticklabels():
    label.set_color('white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
ax.title.set_text('SHAP Mean |SHAP Value| — XGBoost (Top 10)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "shap_bar.png"), dpi=150,
            bbox_inches='tight', facecolor='#1a0a2e')
plt.show()
print("Saved: shap_bar.png")

# ── SHAP Beeswarm ─────────────────────────────────────────────────────────
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  max_display=10, show=False, plot_size=None)
ax = plt.gca()
ax.set_facecolor('#1a0a2e')
plt.gcf().set_facecolor('#1a0a2e')
for label in ax.get_yticklabels():
    label.set_color('#CC99FF')
    label.set_fontsize(12)
for label in ax.get_xticklabels():
    label.set_color('white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
ax.title.set_text('SHAP Feature Importance — XGBoost (Top 10)')
for ax_obj in plt.gcf().get_axes():
    ax_obj.set_facecolor('#1a0a2e')
    for label in ax_obj.get_yticklabels():
        label.set_color('white')
    if hasattr(ax_obj.yaxis, 'label'):
        ax_obj.yaxis.label.set_color('white')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "shap_beeswarm_purple.png"), dpi=150,
            bbox_inches='tight', facecolor='#1a0a2e')
plt.show()
print("Saved: shap_beeswarm_purple.png")

# ── Feature Importance CSV ────────────────────────────────────────────────
mean_shap = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

mean_shap.to_csv(os.path.join(RESULTS, "shap_feature_importance.csv"), index=False)
print("\nTop 10 SHAP features:")
print(mean_shap.head(10).to_string())

# ══════════════════════════════════════════════════════════════════════════
# DAY 8 SUMMARY — SHAP Analysis on XGBoost
# ══════════════════════════════════════════════════════════════════════════
#
# MODEL: XGBoost (best performing model — F1=0.9679, AUC=0.9914)
# SHAP: TreeExplainer on 500 random test samples
#
# TOP 3 FEATURES:
#   1. network_packets_dst_count  SHAP=2.07  High value → attack (flooding)
#   2. network_packet-size_min    SHAP=2.03  Low value  → attack (malformed)
#   3. network_time-delta_min     SHAP=1.70  High value → attack (burst)
#
# KEY OBSERVATION:
#   Top 3 features are 2-7x more impactful than features 4-10.
#   All align with known IIoT attack signatures — volumetric flooding,
#   malformed packet injection, and burst timing attacks.
#   XGBoost learned genuine interpretable attack patterns, not artifacts.
#   Beeswarm shows clear directionality: high dst packet count (pink)
#   strongly pushes predictions toward attack on the right.
# ══════════════════════════════════════════════════════════════════════════